# Pipeline scratchpad

The same code the scheduler runs, importable from here. The Pipeline Airflow
add-on refreshes `/share/pipeline-airflow/lib` on every start, so this notebook
and the DAG can never disagree about what the merge does.

In the **Pipeline Notebook** add-on everything below works as-is: `lib` is already
on `PYTHONPATH` and the addresses come from the add-on's own options. Elsewhere
(the community JupyterLab, say) the next cell supplies both.


In [ ]:
import os, sys

LIB = "/share/pipeline-airflow/lib"
if LIB not in sys.path:
    sys.path.insert(0, LIB)

# Set by the Pipeline Notebook add-on; fall back to the usual addresses.
SPARK_URL = os.environ.get("SPARK_CONNECT_URL", "sc://172.30.32.1:15002")
GYM_TOKEN = os.environ.get("GYM_TRACKER_API_TOKEN", "")

from trackers_feed import get, read_batch
from trackers_merge import merge_batch
print(SPARK_URL)


## 1. Read a tracker directly

The address is `<prefix>-gym-tracker`, the same hostname the DAG derives — no
published port is involved. The prefix is your add-on repository's hash.


In [ ]:
BASE = "http://6753e04e-gym-tracker:8099"

export = get("gym_tracker", BASE, GYM_TOKEN, "/api/export")
print(export["max_seq"], "seq across", len(export["tables"]), "tables")
sorted(export["tables"])


## 2. Query the lakehouse

What the hourly DAG writes. `data` is kept as a JSON *string* on purpose: both
apps gain columns regularly, and inferring a schema per batch would eventually
produce two batches that disagree about a type.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote(SPARK_URL).getOrCreate()
logs = spark.read.format("delta").load("s3a://lakehouse/gym_tracker/workout_logs")
logs.count()


### Parsing `data`

Give it an explicit schema for the columns you want. Anything absent comes back
null rather than failing, so this survives the apps adding columns.


In [ ]:
SCHEMA = ("id long, ts string, exercise_id long, sets long, reps long, "
          "duration_sec long, hr_avg long, hr_max long")

workouts = (
    logs.where("deleted_at IS NULL")
        .select(F.from_json("data", SCHEMA).alias("w"))
        .select("w.*")
)
workouts.orderBy(F.col("ts").desc()).show(10, truncate=False)


### Reps and heart rate per day


In [ ]:
(workouts
 .withColumn("day", F.substring("ts", 1, 10))
 .groupBy("day")
 .agg(F.sum(F.col("sets") * F.col("reps")).alias("reps"),
      F.round(F.avg("hr_avg"), 1).alias("hr_avg"))
 .orderBy("day")
 .toPandas())


## 3. Try a merge without touching the real tables

`merge_batch` takes the lakehouse root as an argument, so point it somewhere
scratch. This is how to test a change to `trackers_merge.py` against real data
before letting the DAG near it — edit `lib/trackers_merge.py`, restart the
kernel, re-run.


In [ ]:
# merge_batch(spark, "gym_tracker",
#             "s3a://raw/gym_tracker/<a timestamped prefix>",
#             "s3a://lakehouse/_scratch")
